# Модуль 2. Предобработка данных и регрессия на датасете «House Prices»

**Студент:** Дремин Иларион Алексеевич  
**Группа:** АСОиУб-23-2  
**Преподаватель:** Игорь Спирин  
**Репозиторий:** https://github.com/DreminIlarion/ml-course-dremin  
**Дата:** 2026-10-13


## 2. Введение

**Цель:** построить модели регрессии для прогнозирования стоимости жилого дома по характеристикам объекта.

**Постановка задачи:** целевая переменная `SalePrice` является непрерывной, поэтому задача относится к регрессии. В работе сравниваются Linear Regression, Ridge, Lasso и ElasticNet.

План: загрузка → EDA → обработка пропусков и выбросов → Feature Engineering → кодирование и масштабирование → обучение 4 моделей → оценка → интерпретация → функция `predict_price()` → сохранение артефактов.

## 3. Теоретическая часть

### Регрессия
Регрессия предсказывает непрерывную величину. Для линейной модели:

$$\hat y = \beta_0 + \sum_{j=1}^{p}\beta_j x_j.$$

### Регуляризация
Ridge добавляет L2-штраф:

$$\min_\beta \left(\sum_i(y_i-\hat y_i)^2 + \alpha\sum_j\beta_j^2\right).$$

Lasso использует L1-штраф:

$$\min_\beta \left(\sum_i(y_i-\hat y_i)^2 + \alpha\sum_j|\beta_j|\right).$$

ElasticNet сочетает L1 и L2-регуляризацию.

### Пропуски
MCAR — пропуск не зависит от данных; MAR — зависит от наблюдаемых признаков; MNAR — зависит от ненаблюдаемого значения/причины. Для House Prices часть пропусков имеет смысл как отдельная категория («нет объекта»), а не как ошибка.

### Логарифмирование
Для цены используется `log1p(SalePrice)`, что уменьшает влияние сильной правосторонней асимметрии и крупных значений.

### Метрики
RMSE:

$$RMSE=\sqrt{\frac1n\sum_i(y_i-\hat y_i)^2}.$$

MAE:

$$MAE=\frac1n\sum_i|y_i-\hat y_i|.$$

Коэффициент детерминации:

$$R^2=1-\frac{\sum_i(y_i-\hat y_i)^2}{\sum_i(y_i-\bar y)^2}.$$

Кросс-валидация и GridSearchCV используются для подбора гиперпараметров регуляризованных моделей.

## 4. Описание данных

Используется обязательный датасет Kaggle **House Prices: Advanced Regression Techniques**: 1460 объектов в `train.csv`, 1459 объектов в `test.csv`, 79 исходных признаков и целевая переменная `SalePrice` в обучающей выборке.

Перед запуском необходимо скачать `train.csv` и `test.csv` с Kaggle и положить их в `data/`.

In [ ]:
# Импорт библиотек и базовые настройки.
from pathlib import Path
import json
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
DATA_DIR = Path("data")
MODEL_DIR = Path("models")
EXAMPLE_DIR = Path("examples")
for folder in [MODEL_DIR, EXAMPLE_DIR]:
    folder.mkdir(exist_ok=True)

train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"

if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError(
        "Не найдены data/train.csv и/или data/test.csv. "
        "Скачайте Kaggle House Prices и положите оба файла в data/."
    )

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("train:", train_df.shape)
print("test :", test_df.shape)
train_df.head()

In [ ]:
# Первичный обзор структуры и статистик.
display(train_df.info())
display(train_df.describe(include="all").T.head(25))

missing = train_df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].head(20).to_frame("missing_count"))

skewness = train_df.select_dtypes(include=np.number).skew().sort_values(ascending=False)
kurtosis = train_df.select_dtypes(include=np.number).kurtosis().sort_values(ascending=False)
display(pd.DataFrame({"skew": skewness, "kurtosis": kurtosis}).head(15))

## 5. Подготовка данных и EDA

В EDA анализируются распределение `SalePrice`, связь цены с качеством, площадь и возраст дома, а также пропуски. Затем выполняются:
- обработка семантических пропусков;
- медианное заполнение `LotFrontage` по `Neighborhood`;
- обработка остальных пропусков;
- IQR-анализ `GrLivArea` и удаление явных аномалий;
- логарифмирование `SalePrice`;
- Feature Engineering;
- ordinal и one-hot encoding.

In [ ]:
# График 1: распределение целевой переменной.
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(train_df["SalePrice"], bins=40)
ax.set_title("Распределение SalePrice")
ax.set_xlabel("SalePrice")
ax.set_ylabel("Количество")
plt.tight_layout()
plt.savefig(EXAMPLE_DIR / "eda_plots.png", dpi=150)
plt.show()

# График 2: логарифмированное распределение.
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(np.log1p(train_df["SalePrice"]), bins=40)
ax.set_title("Распределение log1p(SalePrice)")
ax.set_xlabel("log1p(SalePrice)")
ax.set_ylabel("Количество")
plt.tight_layout()
plt.show()

# График 3: OverallQual vs SalePrice.
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(train_df["OverallQual"], train_df["SalePrice"], alpha=0.45)
ax.set_title("OverallQual и SalePrice")
ax.set_xlabel("OverallQual")
ax.set_ylabel("SalePrice")
plt.tight_layout()
plt.show()

# График 4: GrLivArea vs SalePrice.
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(train_df["GrLivArea"], train_df["SalePrice"], alpha=0.35)
ax.set_title("GrLivArea и SalePrice")
ax.set_xlabel("GrLivArea")
ax.set_ylabel("SalePrice")
plt.tight_layout()
plt.show()

# График 5: цена по району.
top_neigh = train_df.groupby("Neighborhood")["SalePrice"].median().sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(11, 5))
top_neigh.plot(kind="bar", ax=ax)
ax.set_title("Медианная SalePrice для 15 районов")
ax.set_ylabel("Median SalePrice")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Подготовка копий train/test и объединение для согласованного кодирования.
train = train_df.copy()
test = test_df.copy()

# Семантические пропуски означают отсутствие объекта.
none_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"
]
zero_cols = [
    "GarageYrBlt", "GarageArea", "GarageCars",
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
    "BsmtFullBath", "BsmtHalfBath", "MasVnrArea"
]

for col in none_cols:
    if col in train.columns:
        train[col] = train[col].fillna("None")
        test[col] = test[col].fillna("None")

for col in zero_cols:
    if col in train.columns:
        train[col] = train[col].fillna(0)
        test[col] = test[col].fillna(0)

# LotFrontage заполняем медианой внутри Neighborhood.
for df in [train, test]:
    df["LotFrontage"] = df.groupby("Neighborhood")["LotFrontage"].transform(
        lambda s: s.fillna(s.median())
    )
    df["LotFrontage"] = df["LotFrontage"].fillna(train["LotFrontage"].median())

# Остальные числовые пропуски — медианой, категориальные — модой.
for col in train.columns:
    if col in ["SalePrice", "Id"]:
        continue
    if pd.api.types.is_numeric_dtype(train[col]):
        median_value = train[col].median()
        train[col] = train[col].fillna(median_value)
        if col in test.columns:
            test[col] = test[col].fillna(median_value)
    else:
        mode = train[col].mode(dropna=True)
        fill_value = mode.iloc[0] if len(mode) else "None"
        train[col] = train[col].fillna(fill_value)
        if col in test.columns:
            test[col] = test[col].fillna(fill_value)

# IQR для GrLivArea и дополнительное доменное правило из задания.
q1, q3 = train["GrLivArea"].quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
outlier_mask = (train["GrLivArea"] > upper) | (
    (train["GrLivArea"] > 4000) & (train["SalePrice"] < 300000)
)
print("Выбросов до удаления:", int(outlier_mask.sum()))
train = train.loc[~outlier_mask].copy()

# Логарифмирование целевой переменной.
train["SalePrice_log"] = np.log1p(train["SalePrice"])

print("Размер после очистки:", train.shape)

In [ ]:
# Feature Engineering по требованиям задания.
def add_features(df):
    df = df.copy()
    df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
    df["TotalBath"] = (
        df["FullBath"] + 0.5 * df["HalfBath"]
        + df["BsmtFullBath"] + 0.5 * df["BsmtHalfBath"]
    )
    df["TotalPorch"] = (
        df["OpenPorchSF"] + df["3SsnPorch"] + df["EnclosedPorch"]
        + df["ScreenPorch"] + df["WoodDeckSF"]
    )
    df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
    df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
    df["HasPool"] = (df["PoolArea"] > 0).astype(int)
    df["HasGarage"] = (df["GarageArea"] > 0).astype(int)
    df["Has2ndFloor"] = (df["2ndFlrSF"] > 0).astype(int)
    df["HasFireplace"] = (df["Fireplaces"] > 0).astype(int)
    df["OverallQual_TotalSF"] = df["OverallQual"] * df["TotalSF"]
    return df

train = add_features(train)
test = add_features(test)

# Ordinal encoding качества/состояния.
quality_map = {
    "None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5
}
ordinal_cols = [
    "ExterQual", "ExterCond", "BsmtQual", "BsmtCond",
    "HeatingQC", "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond"
]
for col in ordinal_cols:
    if col in train.columns:
        train[col] = train[col].map(quality_map).fillna(0)
        test[col] = test[col].map(quality_map).fillna(0)

# Остальные категориальные признаки кодируем one-hot.
target_cols = ["SalePrice", "SalePrice_log"]
all_data = pd.concat(
    [train.drop(columns=target_cols), test],
    axis=0,
    ignore_index=True
)
all_encoded = pd.get_dummies(all_data, drop_first=False, dtype=float)

X_all_train = all_encoded.iloc[:len(train)].copy()
X_all_test = all_encoded.iloc[len(train):].copy()

X = X_all_train.drop(columns=["Id"], errors="ignore")
X_test_kaggle = X_all_test.drop(columns=["Id"], errors="ignore")
y = train["SalePrice_log"]

print("Признаков после кодирования:", X.shape[1])
print(X.head())

## 6. Построение и обучение моделей

Для честной оценки исходный `train.csv` делится на обучающую и валидационную части. Масштабирование выполняется через `StandardScaler`. Для Ridge, Lasso и ElasticNet подбираются параметры через `GridSearchCV` на обучающей части.

In [ ]:
# Деление на train/validation.
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)

In [ ]:
# Обучение Linear Regression.
start = time.perf_counter()
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
lr_time = time.perf_counter() - start

# Обучение Ridge с GridSearchCV.
ridge_grid = GridSearchCV(
    Ridge(),
    {"alpha": [0.1, 1.0, 10.0, 50.0, 100.0]},
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)
start = time.perf_counter()
ridge_grid.fit(X_train_scaled, y_train)
ridge_model = ridge_grid.best_estimator_
ridge_time = time.perf_counter() - start

# Обучение Lasso с GridSearchCV.
lasso_grid = GridSearchCV(
    Lasso(max_iter=10000),
    {"alpha": [0.0001, 0.0003, 0.0005, 0.001, 0.003]},
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)
start = time.perf_counter()
lasso_grid.fit(X_train_scaled, y_train)
lasso_model = lasso_grid.best_estimator_
lasso_time = time.perf_counter() - start

# Обучение ElasticNet с GridSearchCV.
enet_grid = GridSearchCV(
    ElasticNet(max_iter=10000),
    {
        "alpha": [0.0003, 0.0005, 0.001, 0.003],
        "l1_ratio": [0.2, 0.5, 0.8]
    },
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)
start = time.perf_counter()
enet_grid.fit(X_train_scaled, y_train)
enet_model = enet_grid.best_estimator_
enet_time = time.perf_counter() - start

print("Ridge best:", ridge_grid.best_params_)
print("Lasso best:", lasso_grid.best_params_)
print("ElasticNet best:", enet_grid.best_params_)

## 7. Оценка качества

Основная оценка выполняется в логарифмическом пространстве, так как модель обучается на `log1p(SalePrice)`. Дополнительно RMSE и MAE переводятся обратно в доллары через `expm1`, чтобы результат был интерпретируемым для бизнеса.

Важно: `data/test.csv` не содержит `SalePrice`, поэтому он не используется для расчёта учебных метрик; для этого применяется отложенная часть исходного `train.csv`.

In [ ]:
# Функция расчёта метрик.
def evaluate_model(name, model):
    pred_log = model.predict(X_val_scaled)
    rmse_log = mean_squared_error(y_val, pred_log, squared=False)
    mae_log = mean_absolute_error(y_val, pred_log)
    r2_log = r2_score(y_val, pred_log)

    y_true_real = np.expm1(y_val)
    pred_real = np.expm1(pred_log)
    rmse_real = mean_squared_error(y_true_real, pred_real, squared=False)
    mae_real = mean_absolute_error(y_true_real, pred_real)
    r2_real = r2_score(y_true_real, pred_real)

    return {
        "model": name,
        "rmse_log": rmse_log,
        "mae_log": mae_log,
        "r2_log": r2_log,
        "rmse_real": rmse_real,
        "mae_real": mae_real,
        "r2_real": r2_real,
    }

models = {
    "Linear Regression": lr_model,
    "Ridge": ridge_model,
    "Lasso": lasso_model,
    "ElasticNet": enet_model,
}
results = pd.DataFrame([evaluate_model(name, model) for name, model in models.items()])
display(results.sort_values("rmse_log"))

In [ ]:
# График предсказаний лучшей регуляризованной модели.
regularized = results[results["model"].isin(["Ridge", "Lasso", "ElasticNet"])]
best_regularized_name = regularized.sort_values("rmse_log").iloc[0]["model"]
best_regularized_model = models[best_regularized_name]
best_pred_log = best_regularized_model.predict(X_val_scaled)
best_pred_real = np.expm1(best_pred_log)
y_val_real = np.expm1(y_val)

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_val_real, best_pred_real, alpha=0.45)
lims = [min(y_val_real.min(), best_pred_real.min()), max(y_val_real.max(), best_pred_real.max())]
ax.plot(lims, lims)
ax.set_xlabel("Фактическая цена")
ax.set_ylabel("Предсказанная цена")
ax.set_title(f"Предсказания: {best_regularized_name}")
plt.tight_layout()
plt.savefig(EXAMPLE_DIR / "predictions_scatter.png", dpi=150)
plt.show()

# Анализ остатков.
residuals = y_val_real - best_pred_real
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(best_pred_real, residuals, alpha=0.4)
ax.axhline(0, linestyle="--")
ax.set_xlabel("Предсказанная цена")
ax.set_ylabel("Остаток")
ax.set_title(f"Анализ остатков: {best_regularized_name}")
plt.tight_layout()
plt.savefig(EXAMPLE_DIR / "residuals.png", dpi=150)
plt.show()

## 8. Интерпретация

Коэффициенты линейных моделей показывают направление и относительную силу связи после масштабирования. У Lasso часть коэффициентов становится ровно нулевой, поэтому модель одновременно выполняет регуляризацию и отбор признаков.

Для практической интерпретации особенно полезно смотреть на `OverallQual`, площади, возраст дома, наличие гаража и созданные взаимодействия. Это статистическая интерпретация модели, а не доказательство причинно-следственных связей.

In [ ]:
# Топ коэффициентов лучшей регуляризованной модели.
coef = pd.Series(best_regularized_model.coef_, index=X_train.columns)
coef_abs = coef.abs().sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
coef.loc[coef_abs.index].sort_values().plot(kind="barh", ax=ax)
ax.set_title(f"20 наиболее значимых коэффициентов: {best_regularized_name}")
ax.set_xlabel("Коэффициент")
plt.tight_layout()
plt.show()

# Для Lasso считаем число выбранных и отброшенных признаков.
lasso_zero = int(np.sum(np.isclose(lasso_model.coef_, 0)))
lasso_nonzero = int(len(lasso_model.coef_) - lasso_zero)
print("Lasso: ненулевых признаков =", lasso_nonzero)
print("Lasso: нулевых признаков   =", lasso_zero)

## 9. Функция предсказания

Функция принимает словарь уже подготовленных признаков. Для удобства демонстрации можно передавать несколько ключевых признаков, а остальные признаки автоматически получают значение 0. В реальном приложении лучше использовать тот же полный preprocessing pipeline, что применялся при обучении.

In [ ]:
# Сохраняем медианы и список признаков для воспроизводимости.
feature_cols = list(X.columns)

def predict_price(house_data, model=best_regularized_model, scaler_obj=scaler):
    row = pd.DataFrame([house_data])
    row = row.reindex(columns=feature_cols, fill_value=0)
    row = row.apply(pd.to_numeric, errors="coerce").fillna(0)
    row_scaled = scaler_obj.transform(row)
    pred_log = model.predict(row_scaled)[0]
    return float(np.expm1(pred_log))

examples = [
    {"OverallQual": 5, "GrLivArea": 1000, "TotalSF": 1500, "YearBuilt": 1960, "GarageCars": 1},
    {"OverallQual": 6, "GrLivArea": 1300, "TotalSF": 1800, "YearBuilt": 1975, "GarageCars": 2},
    {"OverallQual": 7, "GrLivArea": 1600, "TotalSF": 2200, "YearBuilt": 1990, "GarageCars": 2},
    {"OverallQual": 8, "GrLivArea": 2000, "TotalSF": 2700, "YearBuilt": 2000, "GarageCars": 3},
    {"OverallQual": 9, "GrLivArea": 2500, "TotalSF": 3300, "YearBuilt": 2010, "GarageCars": 3},
]

for i, example in enumerate(examples, 1):
    print(f"Пример {i}: ${predict_price(example):,.0f}")

## 10. Сохранение модели

Сохраняются все основные артефакты: четыре обученные модели, scaler, список признаков и метрики. Это позволяет воспроизвести inference без повторного обучения.

In [ ]:
# Сохранение моделей и preprocessing-артефактов.
joblib.dump(lr_model, MODEL_DIR / "lr_model.pkl")
joblib.dump(ridge_model, MODEL_DIR / "ridge_model.pkl")
joblib.dump(lasso_model, MODEL_DIR / "lasso_model.pkl")
joblib.dump(enet_model, MODEL_DIR / "enet_model.pkl")
joblib.dump(scaler, MODEL_DIR / "scaler.pkl")
joblib.dump(best_regularized_model, MODEL_DIR / "best_model.pkl")

with open(MODEL_DIR / "feature_cols.json", "w", encoding="utf-8") as f:
    json.dump(feature_cols, f, ensure_ascii=False, indent=2)

metrics = {}
for _, row in results.iterrows():
    name = row["model"].lower().replace(" ", "_")
    metrics[name] = {k: float(row[k]) for k in [
        "rmse_log", "mae_log", "r2_log", "rmse_real", "mae_real", "r2_real"
    ]}

metrics["ridge"]["alpha"] = float(ridge_grid.best_params_["alpha"])
metrics["lasso"]["alpha"] = float(lasso_grid.best_params_["alpha"])
metrics["elasticnet"]["alpha"] = float(enet_grid.best_params_["alpha"])
metrics["elasticnet"]["l1_ratio"] = float(enet_grid.best_params_["l1_ratio"])
metrics["lasso"]["n_features_selected"] = lasso_nonzero
metrics["lasso"]["n_features_dropped"] = lasso_zero
metrics["best_regularized_model"] = best_regularized_name

with open(MODEL_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("Артефакты сохранены в models/")

## 11. Проверка загрузки с GitHub

После публикации файлов в GitHub можно проверить, что модель и preprocessing-артефакты действительно загружаются из репозитория и дают предсказание.

In [ ]:
# Укажи тот же BASE_URL после загрузки файлов в GitHub.
BASE_URL = "https://raw.githubusercontent.com/DreminIlarion/ml-course-dremin/main/module-2-house-prices"

print("BASE_URL:", BASE_URL)
print("Для проверки raw GitHub в Colab можно использовать requests.get(BASE_URL + '/models/metrics.json').")

## 12. Выводы и рекомендации

В работе реализован полный цикл задачи регрессии: EDA, обработка пропусков и выбросов, логарифмирование цели, Feature Engineering, кодирование, масштабирование, обучение четырёх моделей, GridSearchCV, оценка и сохранение артефактов.

Окончательный выбор модели делается по фактическому `rmse_log` и подтверждается анализом остатков. Для дальнейшего улучшения можно попробовать более богатый Feature Engineering, более аккуратный preprocessing pipeline, расширенный подбор гиперпараметров и ансамблирование.

## 13. Источники

1. Kaggle — House Prices: Advanced Regression Techniques.
2. Scikit-learn — LinearRegression.
3. Scikit-learn — Ridge, Lasso, ElasticNet.
4. Scikit-learn — GridSearchCV.
5. Scikit-learn — StandardScaler и метрики регрессии.
6. Pandas documentation — обработка пропусков и категориальных данных.
7. NumPy documentation — `log1p` и `expm1`.

Ссылки и версии библиотек также зафиксированы в `requirements.txt`.

## 14. Приложение — полный код

Полный рабочий код находится в ячейках выше. Для сохранения в Jupytext можно использовать:

```python
# %% [markdown]
# Модуль 2 — House Prices
```

Все основные этапы специально оставлены отдельными ячейками, чтобы notebook соответствовал структуре задания.